# Bit-flip rate vs memory frequency, `alpha_sq = 3`, `eps_p = 0.1`

**What this sweeps.** The bare memory frequency `w_a`, on the full ATS in
the interaction frame (nonlinearity `sin(phi_a + phi_b)`, not the
parity-protected one). The points are given as `w_a = Omega - shift_a * D_a`,
where `Omega = w_d / 2` and `D_a` is the second-order RWA shift of the memory:
`shift_a = 0` is the uncompensated `w_a = Omega`, `shift_a = 1` the
compensated `Omega - D_a`. The bare buffer frequency and the drive both stay
at `w_d`: only the memory is detuned. (`D_a < 0`, so a larger `shift_a` is a
higher `w_a`.)

**The model.** `build_ats_hamiltonian_interaction_detuned`
(`src/floquet_lindblad/models/ats.py`) is
the full-ATS interaction-frame builder driven at `w_d`, with the compensation
of each mode set separately (`compensated_shift` is its `shift_a = shift_b = 1`
case):

    w_a = w_d / 2 - shift_a * D_a
    w_b = w_d     - shift_b * D_b

The sweep runs over the `shift_a` grid of the points cell at `shift_b = 0`;
the grid contains `0` and `1`. The couplings, `epsilon_d`, the drive at `w_d`
and the block `2*pi / (w_d/2)` do not move along it.

**The comparisons.**

- `compensated_shift`, run in the same job: the buffer is compensated too
  (`w_b = w_d - D_b`). Against the sweep's `shift_a = 1` point, it gives the
  effect of the buffer compensation alone.
- `results/sweep_ref.json`, the uncompensated full-ATS reference that
  `sweep_alpha_compensated.ipynb` also plots. The sweep's `shift_a = 0` point
  is the same Hamiltonian, so it should reproduce that rate. This checks the
  detuned builder.

Every point is solved with `solve_point_ks_safe` (Krylov-Schur on the
one-period propagator, `docs/krylov_schur.md`), using the settings of the
`alpha_sq = 3` band of the ladder in `notebooks/sweeps/sweep_alpha_compensated.ipynb`.
The job fans out to one GPU task per point.

## Running it

1. **Kernel**: the workspace pixi environment,
   `/workspace/.pixi/envs/default/bin/python` (**Python 3.13.x**, `ray 2.56.0`).
   These must match the cluster.
2. **Vault session**, once (lasts about a week):

   ```bash
   export VAULT_ADDR=https://vault.int.alice-bob.com
   vault login -method=oidc role=research_core_user
   ```

3. Run all cells. The first `acr.run` of a session may open an interactive
   Microsoft device-code login for GCS read access. That is expected, and the
   login is cached.

If you have already run something that imports `ray` in this kernel, restart it
first. Ray caches `RAY_AUTH_MODE` at first import.

In [ ]:
import os
import subprocess
import sys

assert sys.version_info[:2] == (3, 13), (
    f"kernel runs Python {sys.version.split()[0]}; the cluster needs 3.13.x. "
    "Select the workspace pixi environment as the kernel."
)

# Ray reads RAY_AUTH_MODE once, at its first import, and caches it. Setting it
# here works *only* because nothing has imported ray yet in a fresh kernel.
os.environ["RAY_AUTH_MODE"] = "token"
os.environ.setdefault("VAULT_ADDR", "https://vault.int.alice-bob.com")

if not os.environ.get("RAY_AUTH_TOKEN"):
    try:
        os.environ["RAY_AUTH_TOKEN"] = subprocess.run(
            ["vault", "kv", "get", "-field=token", "kv-it/prod/ray/auth-token"],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError) as exc:
        raise RuntimeError(
            "Could not mint the cluster token from Vault. In a terminal, run:\n"
            "  export VAULT_ADDR=https://vault.int.alice-bob.com\n"
            "  vault login -method=oidc role=research_core_user\n"
            "then re-run this cell (no kernel restart needed).\n"
            f"{(getattr(exc, 'stderr', '') or '').strip()}"
        ) from exc

import numpy as np

from anb_compute import ray as acr

import pathlib


def _find_repo():
    """Locate the repo checkout without hardcoding an absolute path.

    Notebooks have no __file__, so search upward from the working directory --
    which VS Code sets to either the notebook's folder or the workspace root,
    depending on `jupyter.notebookFileRoot`. Both are handled.
    """
    here = pathlib.Path.cwd().resolve()
    for base in (here, *here.parents):
        if (base / "src" / "floquet_lindblad").is_dir():
            return base
        for hit in sorted(base.glob("*/src/floquet_lindblad")):
            return hit.parents[1]
    raise RuntimeError(f"no checkout with src/floquet_lindblad found above {here}")


REPO = _find_repo()

# Shipped by PATH, not by name: a bare module name that resolves to a
# non-editable site-packages install ships a frozen copy, and local edits would
# never reach the worker. These prepend to the worker's sys.path, so the
# dynamiqs here shadows the image's stock 0.3.6 (which lacks `mesolve_fast`).
PKG = str(REPO / "src" / "floquet_lindblad")
# The patched dynamiqs checkout, expected beside the repo; $DYNAMIQS_SRC wins.
DYNAMIQS = os.environ.get("DYNAMIQS_SRC") or str(
    REPO.parent / "dynamiqs" / "dynamiqs"
)
for _p in (PKG, DYNAMIQS):
    if not pathlib.Path(_p).is_dir():
        raise RuntimeError(f"not a directory, cannot ship to workers: {_p}")

RUNTIME_ENV = acr.build_runtime_env(py_modules=[PKG, DYNAMIQS])

print("ready")

### The points

`D_a` depends only on `eps_p` and the circuit constants, and those live in
`ats.py`, which this client cannot import. So the points are given as
`shift_a`, and the builder turns each one into `w_a` on the worker. The
resulting `w_a`, `w_b`, `D_a` and `D_b` come back in each point's `params`.

In [ ]:
import numpy as np

ALPHA_SQ = 3.0
EPS_P = 0.1

# w_a = Omega - shift_a * D_a: shift_a = 0 -> Omega (uncompensated),
# shift_a = 1 -> Omega - D_a (compensated). Both must stay in the grid: they are
# the two comparison points.

SHIFT_A_LIST = sorted({*np.linspace(1.5, 0.0, 20).tolist(), *np.linspace(1.1, 0.85, 10).tolist(), 1.0}, reverse=True)
SHIFT_B_SWEEP = 0.0  # bare buffer at w_d along the sweep

# The alpha_sq < 3.5 band of the ladder in sweep_alpha_compensated.ipynb,
# where this alpha_sq passed the truncation check (top-3 tail ~6e-12 < 1e-11 in
# notebooks/solvers/compensated_shift_krylov_schur.ipynb).
SETTINGS = dict(n_a=25, n_b=12, m=40, k=10,
                rtol_loop=1e-8, atol_loop=1e-9, res_tol_loop=1e-7,
                rtol_final=1e-10, atol_final=1e-11, res_tol=1e-9,
                n_blocks_final=20, max_cycles=30)

POINTS = (
    [dict(label=f"sweep {i}", model="detuned", shift_a=float(s), shift_b=SHIFT_B_SWEEP)
     for i, s in enumerate(SHIFT_A_LIST)]
    + [dict(label="compensated_shift", model="compensated_shift")]
)
assert any(np.isclose(s, 0.0) for s in SHIFT_A_LIST) and any(
    np.isclose(s, 1.0) for s in SHIFT_A_LIST), "keep shift_a = 0 and 1 in the grid"

COMMON = dict(alpha_sq=ALPHA_SQ, eps_p=EPS_P, seed=0, **SETTINGS)

print(f"alpha_sq = {ALPHA_SQ}, eps_p = {EPS_P}, n_a x n_b = "
      f"{SETTINGS['n_a']} x {SETTINGS['n_b']}, {len(POINTS)} GPU task(s)\n")
print(f"{'label':<18} {'model':<18} {'shift_a':>8} {'shift_b':>8}")
for p in POINTS:
    print(f"{p['label']:<18} {p['model']:<18} "
          f"{p.get('shift_a', 1.0):8.3f} {p.get('shift_b', 1.0):8.3f}")

### One point, on one GPU worker

`floquet_lindblad` is imported inside the function, on the worker. The
detuned builder is handed to the pipeline as a callable, with the two shifts
bound. The pipeline calls it with the registry signature
(`n_a, n_b, alpha_sq, epsilon_p, n_periods`).

In [ ]:
def solve_point(*, label, model, shift_a=1.0, shift_b=1.0, **kwargs):
    """`floquet_lindblad.pipeline.solve_point_ks_safe` on one model, on the worker.

    `model="compensated_shift"` runs the registry builder as it is;
    `model="detuned"` runs `build_ats_hamiltonian_interaction_detuned` at the
    given shifts. The label and the shifts are copied onto the result (and onto
    an error entry, which has no `params`).
    """
    from floquet_lindblad.pipeline import solve_point_ks_safe as solve

    if model == "compensated_shift":
        res = solve(hamiltonian="compensated_shift", **kwargs)
    elif model == "detuned":
        from floquet_lindblad.models import Lindbladian
        from floquet_lindblad.models.ats import (
            build_ats_hamiltonian_interaction_detuned as build,
        )

        def detuned(**kw):
            H, jump_ops, jump_ops_LdL, output_phase, V, T_block, params = build(
                shift_a=shift_a, shift_b=shift_b, **kw
            )
            return Lindbladian(H, jump_ops, T_block, params,
                               jump_ops_LdL, output_phase, V)

        detuned.__name__ = "detuned"
        res = solve(hamiltonian=detuned, **kwargs)
    else:
        raise ValueError(f"unknown model {model!r}")

    res.update(label=label, model=model, shift_a=float(shift_a), shift_b=float(shift_b))
    return res

### The sweep

In [ ]:
import time


def sweep(worker, points, common):
    """Coordinator: fans one GPU task out per point, gathers as they land.

    Runs as the driver on a worker; each task asks for its own GPU. `worker` is
    passed in because it is defined in this notebook and ships by value.
    """
    import ray

    from anb_compute import ray as acr

    refs = {
        acr.submit(worker, num_gpus=.5, num_cpus=2, **common, **p): p
        for p in points
    }

    out = []
    pending = list(refs)
    while pending:
        ready, pending = ray.wait(pending, num_returns=1)
        p = refs[ready[0]]
        try:
            res = ray.get(ready[0])
        except Exception as exc:  # a task lost to infra, not to numerics
            res = {**p, "error": f"{type(exc).__name__}: {exc}"}
        out.append(res)
        print(f"{p['label']:<18} "
              + (res["error"] if "error" in res
                 else f"rate_bf = {res['rate_bf'].real:.6e}  "
                      f"res = {res['res_rel']:.1e}  "
                      f"apps = {res['n_apply']:d}"
                      + ("" if res["converged"] else "  (NOT converged)")))
    order = {p["label"]: i for i, p in enumerate(points)}
    out.sort(key=lambda d: order[d["label"]])
    return out


t0 = time.time()
results = acr.run(
    sweep, solve_point, POINTS, COMMON,
    runtime_env=RUNTIME_ENV,
    poll_seconds=5,
    # No resources here on purpose: the fan-out inside asks for its own.
)
print(f"\nwall clock, incl. job startup: {time.time() - t0:.1f} s "
      f"for {len(results)} point(s)")

### Results

Frequencies are angular, in the units of `ats.W_A`, so `/2pi` gives GHz and
`x 1e3 / 2pi` gives MHz. The offsets are taken from the resonant values:
`w_a - w_d/2` and `w_b - w_d`.

In [ ]:
ok = [r for r in results if "error" not in r]
bad = [r for r in results if "error" in r]

TO_MHZ = 1e3 / (2 * np.pi)


def offsets(r):
    """(w_a - w_d/2, w_b - w_d) of a point, in MHz."""
    p = r["params"]
    return (p["w_a"] - p["w_d"] / 2) * TO_MHZ, (p["w_b"] - p["w_d"]) * TO_MHZ


p0 = ok[0]["params"]
print(f"w_d/2pi = {p0['w_d'] / (2 * np.pi):.6f} GHz   "
      f"D_a/2pi = {p0['D_a'] * TO_MHZ:+.4f} MHz   D_b/2pi = {p0['D_b'] * TO_MHZ:+.4f} MHz   "
      f"kappa_2/2pi = {p0['kappa_2'] * TO_MHZ:.4f} MHz\n")

print(f"{'label':<18} {'shift_a':>7} {'shift_b':>7} {'dw_a MHz':>9} {'dw_b MHz':>9} "
      f"{'rate_bf':>13} {'res Krylov':>11} {'res RQ':>9} {'blocks':>7} {'apps':>5} "
      f"{'conv':>5} {'GPU s':>6}")
for r in ok:
    dwa, dwb = offsets(r)
    print(f"{r['label']:<18} {r['shift_a']:7.3f} {r['shift_b']:7.3f} {dwa:9.4f} {dwb:9.4f} "
          f"{r['rate_bf'].real:13.6e} {r['res_rel']:11.2e} {r['res_rel_n']:9.2e} "
          f"{r['n_blocks_final']:7d} {r['n_apply']:5d} "
          f"{('yes' if r['converged'] else 'NO'):>5} "
          f"{r['timings_s']['loop'] + r['timings_s']['polish']:6.1f}")
for r in bad:
    print(f"{r['label']:<18}  FAILED: {r['error']}")

### Comparisons

- **Builder check.** The sweep at `shift_a = 0` (`w_a = w_d/2`, `w_b = w_d`) is
  the uncompensated full ATS, the Hamiltonian of `results/sweep_ref.json`. The
  two rates should agree to the solver precision (about 1e-4 relative between
  runs with these tolerances).
- **Buffer shift.** The sweep at `shift_a = 1` (the compensated memory, buffer
  left at `w_d`) against `compensated_shift` (buffer at `w_d - D_b` too). Both
  points have the same `w_a`, so the ratio says how much the buffer
  compensation matters on its own.


In [ ]:
import json

import matplotlib.pyplot as plt

by_label = {r["label"]: r for r in ok}
sweep_pts = [r for r in ok if r["label"].startswith("sweep")]
comp = by_label.get("compensated_shift")


def _complex_hook(d):
    """{"real", "imag"} -> complex, as floquet_lindblad.io.from_jsonable does
    (that module imports jax, which this client does not have)."""
    return complex(d["real"], d["imag"]) if d.keys() == {"real", "imag"} else d


# The uncompensated full-ATS reference at this (alpha_sq, eps_p), by true cat size.
with open(REPO / "results" / "sweep_ref.json") as f:
    ref = [r for r in json.load(f, object_hook=_complex_hook)
           if "error" not in r and np.isclose(r["eps_p"], EPS_P)
           and np.isclose(r["params"]["epsilon_d"] / (2 * r["params"]["g"]), ALPHA_SQ)]
rate_ref = ref[0]["rate_bf"].real if ref else None

dwa = np.array([offsets(r)[0] for r in sweep_pts])
rate = np.array([r["rate_bf"].real for r in sweep_pts])
conv = np.array([r["converged"] for r in sweep_pts])
D_a_mhz = p0["D_a"] * TO_MHZ  # w_a = Omega - D_a is at dw_a = -D_a

at_zero = [r for r in sweep_pts if np.isclose(r["shift_a"], 0.0)]
if rate_ref is None:
    print(f"no point at alpha_sq = {ALPHA_SQ}, eps_p = {EPS_P} in sweep_ref.json: no builder check")
elif at_zero:
    rate_zero = at_zero[0]["rate_bf"].real
    print(f"builder check: sweep at shift_a = 0 / sweep_ref.json = "
          f"{rate_zero:.6e} / {rate_ref:.6e} = {rate_zero / rate_ref:.6f}  "
          f"(rel. dev. {abs(rate_zero / rate_ref - 1):.1e})\n")

if comp is not None:
    rate_comp = comp["rate_bf"].real
    at_comp = [r for r in sweep_pts if np.isclose(r["shift_a"], 1.0)]
    if at_comp:
        rate_sweep_at_comp = at_comp[0]["rate_bf"].real
        print(f"sweep (w_b = w_d) at w_a = Omega - D_a ({at_comp[0]['label']}): "
              f"{rate_sweep_at_comp:.6e}")
        print(f"compensated_shift (w_b = w_d - D_b):                 {rate_comp:.6e}")
        print(f"ratio sweep / compensated at the compensated w_a:    "
              f"{rate_sweep_at_comp / rate_comp:.4f}")
    else:
        print("the shift_a = 1 point of the sweep failed: no buffer-shift ratio")
    i_min = int(np.argmin(rate))
    print(f"\nlowest rate along the sweep: {rate[i_min]:.6e} at dw_a = {dwa[i_min]:.4f} MHz "
          f"(shift_a = {sweep_pts[i_min]['shift_a']:.3f}), "
          f"{rate[i_min] / rate_comp:.4f} x compensated_shift")

BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8c8c8c"
fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.0), constrained_layout=True)

ax = axes[0]
ax.plot(dwa, rate, "-", lw=1.0, color=BLUE, zorder=2)
ax.scatter(dwa[conv], rate[conv], s=30, color=BLUE, zorder=3,
           label=r"sweep, $\omega_b = \omega_d$")
if (~conv).any():
    ax.scatter(dwa[~conv], rate[~conv], s=30, facecolors="white", edgecolors=BLUE,
               zorder=3, label="not converged")
if comp is not None:
    ax.scatter([offsets(comp)[0]], [rate_comp], s=90, marker="*", color=ORANGE, zorder=4,
               label=r"compensated_shift, $\omega_b = \omega_d - D_b$")
if rate_ref is not None:
    ax.scatter([0.0], [rate_ref], s=70, marker="o", facecolors="none",
               edgecolors=GRAY, lw=1.2, zorder=5,
               label="sweep_ref.json (uncompensated), builder check")
ax.axvline(-D_a_mhz, color="0.4", lw=0.8, ls=":")
ax.annotate(r"$\omega_d/2 - D_a$", (-D_a_mhz, 0), xycoords=("data", "axes fraction"),
            xytext=(3, 4), textcoords="offset points", fontsize=8, color="0.3")
ax.set_yscale("log")
ax.set_xlabel(r"$(\omega_a - \omega_d/2)\,/\,2\pi$  (MHz)")
ax.set_ylabel("bit-flip rate (GHz)")
ax.set_title(rf"$|\alpha|^2$ = {ALPHA_SQ:g}, $\varepsilon_p$ = {EPS_P:g}", fontsize=9)
ax.legend(fontsize=8, frameon=False)

ax = axes[1]
if comp is not None:
    ax.plot(dwa, rate / rate_comp, ".-", ms=8, lw=1.0, color=BLUE,
            label="sweep / compensated_shift")
    ax.axhline(1.0, color="k", lw=0.8, ls=":")
    ax.axvline(-D_a_mhz, color="0.4", lw=0.8, ls=":")
    ax.legend(fontsize=8, frameon=False)
ax.set_xlabel(r"$(\omega_a - \omega_d/2)\,/\,2\pi$  (MHz)")
ax.set_ylabel("rate ratio")

for ax in axes:
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.grid(True, which="major", lw=0.4, color="0.9")
    ax.set_axisbelow(True)
plt.show()

### Truncation check

Detuning the memory changes the cat a little, so the top-level Fock weight is
checked at every point. The threshold is the same as in the other notebooks.

In [ ]:
TOL = 1e-11

print(f"{'label':<18} {'tail_a':>10} {'tail_b':>10} {'res RQ':>10} {'1-|mu|':>10} {'verdict':>8}")
failed = []
for r in ok:
    passed = max(r["tail_a"], r["tail_b"]) < TOL
    if not passed:
        failed.append(r["label"])
    print(f"{r['label']:<18} {r['tail_a']:10.2e} {r['tail_b']:10.2e} "
          f"{r['res_rel_n']:10.2e} {1 - abs(r['mu']):10.2e} "
          f"{('PASS' if passed else 'FAIL'):>8}")
print(f"\ntop-{ok[0]['k_top']} weight against tol = {TOL:.0e}: "
      + ("all points PASS" if not failed else "raise n_a / n_b for " + ", ".join(failed)))

### Save

In [ ]:
import json


def to_jsonable(obj):
    """numpy-only copy of floquet_lindblad.io.to_jsonable (that module imports jax,
    which this client does not have). Complex -> {"real", "imag"}, arrays -> lists."""
    if isinstance(obj, dict):
        return {k: to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return to_jsonable(obj.tolist())
    if isinstance(obj, (np.complexfloating, complex)):
        return {"real": float(obj.real), "imag": float(obj.imag)}
    if isinstance(obj, (np.floating, float)):
        return float(obj)
    if isinstance(obj, (np.bool_, bool)):
        return bool(obj)
    if isinstance(obj, (np.integer, int)):
        return int(obj)
    return obj


out_path = REPO / "results" / "sweep_w_a_detuned.json"
out_path.parent.mkdir(exist_ok=True)
with open(out_path, "w") as f:
    json.dump(to_jsonable(results), f, indent=2)
print(f"wrote {len(results)} points to {out_path}")